In [ ]:
from pathlib import Path
import runpy

def _find_notebook_bootstrap(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "notebooks" / "_bootstrap.py"
        if direct.exists():
            return direct
        nested = candidate / "abstractgraph-ml" / "notebooks" / "_bootstrap.py"
        if nested.exists():
            return nested
    raise FileNotFoundError("Could not locate notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_find_notebook_bootstrap(Path.cwd())))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import scipy as sp
import pandas as pd
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt
from abstractgraph.utils import plot_dataset_method_bars, plot_pareto
from IPython.core.display import HTML
HTML('<style>.container { width:95% !important; }</style><style>.output_png {display: table-cell;text-align: center;vertical-align: middle;}</style>')

In [ ]:
import time
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
def perf(neural_graph_estimator, graphs, targets):    
    # Train metrics
    t_pred0 = time.perf_counter()
    pred = neural_graph_estimator.predict(graphs)
    try:
        proba = neural_graph_estimator.predict_proba(graphs)
    except Exception:
        proba = None
    t_pred1 = time.perf_counter()
    pred_time = t_pred1 - t_pred0
    acc = accuracy_score(targets, pred)
    errors = int((pred != targets).sum())
    if proba is not None and proba.ndim == 2 and proba.shape[1] > 1:
        if proba.shape[1] == 2:
            auc = roc_auc_score(targets, proba[:, 1])
            ap = average_precision_score(targets, proba[:, 1])
        else:
            auc = roc_auc_score(targets, proba, average='macro', multi_class='ovr')
            ap = average_precision_score(targets, proba, average='macro')
    else:
        auc, ap = None, None
    return acc, auc, ap, errors, pred_time

def print_perf(header_str, acc, auc, ap, errors, fit_time, pred_time, n_instances):
    n_instances_per_second = n_instances / (fit_time+pred_time)
    print(f"{header_str}: accuracy={acc:.3f}, roc_auc={(auc if auc is not None else float('nan')):.3f}, avg_precision={(ap if ap is not None else float('nan')):.3f}, errors={errors}, fit_time={fit_time:.2f}s, pred_time={pred_time:.2f}s, total_time={(fit_time+pred_time):.2f}s, n_instances_per_second={n_instances_per_second:.1f}  ")

## Dataset info & examples

In [ ]:
from coco_grape.visualizer.mol_display import draw_molecules
from coco_grape.data_loader.mol.mol_loader import PubChemLoader
#assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_ids = [2631]
assay_ids = ['488975'] #active 2,634
datasets = []
for assay_id in assay_ids:
    def pubchem_loader(): return PubChemLoader().load(assay_id)
    from coco_grape.data_loader.loader import SupervisedDataSetLoader
    original_graphs, original_targets = SupervisedDataSetLoader(pubchem_loader, size=1e6, use_equalized=True).load()
    print(f'AID{assay_id}  #graphs: {len(original_graphs)}')
    datasets.append((assay_id, original_graphs, original_targets))

In [ ]:
from abstractgraph.operators import *
nbits = 14

def make_nsppk(radius=1, distance=3, connector=1):
    if connector == 0:
        df = add(union_of_shortest_paths(length=distance), compose(combination(number_of_elements=2,distance=(0,distance)), neighborhood(radius=(0,radius))))
    elif connector == 1:
        df = compose(combination(number_of_elements=2,distance=(0,distance)), neighborhood(radius=(0,radius)))
    else:
        raise Exception(f'connector {connector} is not allowed')
    return df
df = make_nsppk(radius=1, distance=4, connector=1)

paired_neighs = compose(combination(number_of_elements=2,distance=(3,4)), neighborhood(radius=(0,2)))
cycle_and_tree = (cycle(), compose(neighborhood(radius=1), tree()))
df = add(paired_neighs, *cycle_and_tree, compose_product(binary_combination(distance=0), *cycle_and_tree))

In [ ]:
from abstractgraph.display import display_decomposition_graph
print('graph base decomposition')
display_decomposition_graph(df)

In [ ]:
from abstractgraph.vectorize import AbstractGraphTransformer
transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=df,
    return_dense=True,
    n_jobs=-1,
)

from sklearn.ensemble import RandomForestClassifier
from abstractgraph_ml.estimators import GraphEstimator
graph_estimator = GraphEstimator(
    transformer=transformer,
    estimator=RandomForestClassifier(random_state=0, n_estimators=300, n_jobs=-1),
    manifold=None,
    n_selected_features=500,
)

In [ ]:
from sklearn.model_selection import train_test_split
import time

assay_id, graphs, targets = datasets[0]
n_instances = len(graphs)
targets = np.array(targets)
# Train/test split (scikit-learn)
graphs_tr, graphs_te, targets_tr, targets_te = train_test_split(graphs, targets, test_size=0.2, random_state=0)
class_counts = dict(zip(*np.unique(targets, return_counts=True)))
print(f'assay_id:{assay_id}   #graphs:{n_instances}   #train:{len(graphs_tr)}   #test:{len(graphs_te)} (class_counts={class_counts})')

t_fit0 = time.perf_counter()
graph_estimator.fit(graphs_tr, targets_tr)
t_fit1 = time.perf_counter()
fit_time = t_fit1 - t_fit0
acc_te, auc_te, ap_te, errors_te, pred_time_te = perf(graph_estimator, graphs_te, targets_te)
print_perf('Test ', acc_te, auc_te, ap_te, errors_te, fit_time, pred_time_te, len(graphs))

In [ ]:
from abstractgraph_ml.importance import plot_graph_node_saliency_with_estimator
import random
_ = plot_graph_node_saliency_with_estimator(
    graphs_te[random.randrange(len(graphs_te))], graph_estimator,
    node_agg="max", #node_agg: Aggregation strategy over interpretation nodes ("max", "mean", "sum").
    edge_stat="mean", #edge_stat: Edge aggregation ("mean", "min", "max").
    cmap="YlOrRd",
    size=(6, 6)
    )

In [ ]:
# compute probabilities
proba_te = graph_estimator.predict_proba(graphs_te)
classes = graph_estimator.estimator_.classes_
class_to_col = {cls: i for i, cls in enumerate(classes)}

In [ ]:
from abstractgraph_ml.importance import plot_graph_node_saliency_grid
# select a random sample of n_show elements per class 
# sampling proportionally to the prob for their respective class
n_show = 14
n_elements_per_row = 7

selected = []
for cls in classes:
    idxs = np.where(targets_te == cls)[0]
    col = class_to_col[cls]
    p = proba_te[idxs, col]
    p = p / p.sum()
    size = min(n_show, idxs.size)
    top = np.random.choice(idxs, size=size, replace=False, p=p)
    top = top[np.argsort(-proba_te[top, col])]
    selected.append((cls, col, top))

# plot the importance of each node/edge
for cls, col, top in selected:
    print('_'*180)
    titles = [f'p={proba_te[i, col]:.3f}' for i in top]
    sel_graphs = [graphs_te[i] for i in top]
    _ = plot_graph_node_saliency_grid(
        sel_graphs,
        graph_estimator=graph_estimator,
        n_elements_per_row=n_elements_per_row,
        suptitle=f'class={cls}',
        titles=titles,
        node_agg="mean", #node_agg: Aggregation strategy over interpretation nodes ("max", "mean", "sum").
        edge_stat="min", #edge_stat: Edge aggregation ("mean", "min", "max").
        figsize_per_graph=(4.5,4.5)
    )

---